# 🎤 Karaoke Maker V2 — Google Colab

<div dir="rtl">

**הפעלה בלי להתקין כלום על המחשב.**

1. בתפריט למעלה: **Runtime → Change runtime type → T4 GPU → Save** (פעם אחת).
2. לחץ ▶ על **שלב 1** וחכה שיופיע ✅ (בערך 5–10 דקות).
3. לחץ ▶ על **שלב 2** ולחץ על הקישור **🎤 Open Karaoke Maker** שמופיע.

השאר את הלשונית של Colab פתוחה בזמן העבודה, והורד כל סרטון מוכן מיד, כי Colab מוחק הכל כשהשרת נסגר.

</div>

In [ ]:
#@title שלב 1 · התקנה (פעם אחת בכל פתיחה, 5–10 דקות)
import os

REPO = "https://github.com/regevgiambasho/Kareoke-Maker-RG"
APP_DIR = "/content/karaoke"

!nvidia-smi -L 2>/dev/null || echo "⚠️ No GPU: Runtime > Change runtime type > T4 GPU (works without GPU, but slower)"

print("Installing system packages…")
!apt-get -qq update > /dev/null && apt-get -qq install -y ffmpeg fonts-dejavu-core fontconfig libsndfile1 > /dev/null

print("Downloading the app…")
if os.path.isdir(APP_DIR):
    !git -C {APP_DIR} pull -q
else:
    !git clone -q --depth 1 {REPO} {APP_DIR}

print("Installing Python packages (the longest step)…")
!pip install -q -r {APP_DIR}/requirements.txt 2>&1 | grep -viE "dependency conflicts|incompatible|^\s*$" | tail -n 20

missing = [t for t in ("ffmpeg", "ffprobe", "audio-separator") if not __import__("shutil").which(t)]
if missing:
    print("❌ Missing after install:", ", ".join(missing))
else:
    print("✅ Installation finished. Run step 2.")

In [ ]:
#@title שלב 2 · הפעלת האפליקציה
import os, subprocess, time, urllib.request

APP_DIR = "/content/karaoke"
LOG = "/content/karaoke_app.log"

if "app_proc" in globals() and app_proc.poll() is None:
    app_proc.terminate()
    app_proc.wait()

app_proc = subprocess.Popen(
    ["python", "launcher.py"],
    cwd=APP_DIR,
    stdout=open(LOG, "w"),
    stderr=subprocess.STDOUT,
)

ready = False
for _ in range(120):
    if app_proc.poll() is not None:
        break
    try:
        ready = urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health", timeout=2).read() == b"ok"
    except Exception:
        ready = False
    if ready:
        break
    time.sleep(1)

if ready:
    from google.colab import output
    print("✅ The app is running. Click the link (opens in a new tab):")
    output.serve_kernel_port_as_window(8501, anchor_text="🎤 Open Karaoke Maker")
else:
    print("❌ The app did not start. Last log lines:\n")
    print("".join(open(LOG).readlines()[-40:]))

In [ ]:
#@title (אם משהו לא עובד) הצגת יומן האפליקציה
print("".join(open("/content/karaoke_app.log").readlines()[-80:]))